# 06 — CI methods comparison

Three CI methods are computed from every bootstrap run:

- **percentile** — `[P2.5, P97.5]` of the bootstrap distribution. Biased under skew.
- **Fisher-z** — normal-approx CI on the $z$-space, back-transformed via tanh. Symmetric in $z$, asymmetric in $r$.
- **BCa** — bias-corrected and accelerated. Adjusts the percentile quantiles using a bias estimate $\hat z_0$ and a jackknife-based acceleration $\hat a$.

See STATS.md §4 for the equations. **Recommended headline: BCa.**

This notebook does two things:
1. **Synthetic-truth coverage check.** Generate many replicates with known true $\rho$; estimate the empirical coverage rate of each CI.
2. **Real-data comparison.** Run a single intergroup correlation on your actual data and visualize all three CIs side by side, both for participant- and stimulus-bootstrap.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

HERE = Path.cwd(); ROOT = HERE.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from python import (
    calculate_split_half_reliability,
    bootstrap_intergroup_correlation_sem,
    load_group,
)

## Part 1 — Coverage under known truth

Generates `n_reps` replicates of two groups with a fixed true $\rho$. For each replicate, runs the bootstrap and records whether each CI method's interval covered $\rho$. The empirical coverage rate should be $\approx 1-\alpha = 0.95$.

Set `RHO_TRUE` (and the rest of the synthetic-data knobs) at the top; `N_REPS=100` runs in ~1 min on a laptop.

In [ ]:
RHO_TRUE   = 0.5
N_A, N_B   = 25, 25
N_ITEMS    = 50
TARGET_REL = 0.7   # within-group reliability you want to simulate
BASE_RATE  = 0.7
ITEM_SPREAD= 0.15
N_REPS     = 100
N_BOOT     = 800
N_SPLITS   = 200
ALPHA      = 0.05
TARGET     = 'point_corr'   # 'point_corr' (corrected) or 'point_raw'

In [ ]:
def synth_pair(rho, n_a, n_b, n_items, target_rel, base_rate, item_spread, rng):
    R = np.array([[1, rho], [rho, 1]])
    Z = rng.multivariate_normal([0, 0], R, size=n_items)
    item_rates = np.clip(base_rate + item_spread * Z, 0.02, 0.98)
    def mk(idx, n):
        sig_e = item_spread * np.sqrt(n * (1 - target_rel) / target_rel) if 0 < target_rel < 1 else 0.0
        E = sig_e * rng.standard_normal((n, n_items))
        P = np.clip(item_rates[:, idx][None, :] + E, 0.01, 0.99)
        return (rng.random((n, n_items)) < P).astype(float)
    return mk(0, n_a), mk(1, n_b)

rng = np.random.default_rng(0)
items = np.array([f'item{i:03d}' for i in range(N_ITEMS)], dtype=object)

covered = {'percentile': 0, 'fisher_z': 0, 'bca': 0}
widths  = {'percentile': [], 'fisher_z': [], 'bca': []}

for rep in range(N_REPS):
    XA, XB = synth_pair(RHO_TRUE, N_A, N_B, N_ITEMS, TARGET_REL, BASE_RATE, ITEM_SPREAD, rng)
    oa = calculate_split_half_reliability(XA, np.full_like(XA, np.nan), items, n_splits=N_SPLITS, rng=rng)
    ob = calculate_split_half_reliability(XB, np.full_like(XB, np.nan), items, n_splits=N_SPLITS, rng=rng)
    res = bootstrap_intergroup_correlation_sem(oa, ob, 'hit', n_boot=N_BOOT, rng=rng, verbose=False)
    cis = res.cis_corr if TARGET == 'point_corr' else res.cis_raw
    for k, (lo, hi) in cis.items():
        if np.isfinite(lo) and np.isfinite(hi):
            if lo <= RHO_TRUE <= hi:
                covered[k] += 1
            widths[k].append(hi - lo)
    if (rep + 1) % 20 == 0:
        print(f'  rep {rep+1}/{N_REPS}')

summary = pd.DataFrame({
    'method'   : list(covered.keys()),
    'coverage' : [covered[k]/N_REPS for k in covered],
    'mean width': [float(np.mean(widths[k])) if widths[k] else float('nan') for k in covered],
})
summary['target'] = f'{1-ALPHA}'
print(); print(summary.to_string(index=False))
print(f'\nTRUE rho = {RHO_TRUE}; target coverage = {1-ALPHA}')
print('Methods that are below 0.95 are under-covering (CI too narrow / biased).')
print('Methods substantially above 0.95 are over-covering (CI too wide).')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(range(3), summary['coverage'], color=['#888','#4060b0','#b04040'])
axes[0].axhline(1-ALPHA, color='k', lw=1, ls='--', alpha=0.7, label='target = 0.95')
axes[0].set_xticks(range(3)); axes[0].set_xticklabels(summary['method'])
axes[0].set_ylabel('empirical coverage'); axes[0].set_ylim(0, 1.05); axes[0].legend()
axes[0].set_title(f'Coverage at true rho = {RHO_TRUE}')
axes[1].bar(range(3), summary['mean width'], color=['#888','#4060b0','#b04040'])
axes[1].set_xticks(range(3)); axes[1].set_xticklabels(summary['method'])
axes[1].set_ylabel('mean CI width'); axes[1].set_title('Average CI width')
plt.tight_layout(); plt.show()

**Reading the coverage plot.**

- If percentile coverage falls below 0.95 while BCa stays at 0.95, BCa's bias correction was load-bearing — the bootstrap was skewed and BCa fixed it.
- If all three are at 0.95, your bootstrap is well-behaved at this N and you could report any of them.
- If Fisher-z is much wider than the others, the bootstrap z-space SD is dominating — likely a few extreme bootstrap draws.

Try other `RHO_TRUE` values (0.2, 0.8) — coverage typically degrades for percentile when $|\rho|$ is large (the sampling distribution of $r$ gets more skewed near the boundary).

## Part 2 — Real data, side by side

Load US, San Borja, Tsimane' from your `.mat` files. For each pair, compute both **participant-bootstrap** and **stimulus-bootstrap** CIs under all three methods. Plot them side by side so you can see (a) which CI method shifts the bars, and (b) whether participant or stimulus variance is dominant.

In [ ]:
BASE_DIR  = ROOT.parents[2] / 'Data' / 'RecognitionMemory' / 'Results'
CONDITION = 'Globalized-Music'
TRIAL     = 'hit'
MIN_RESP  = 2
N_BOOT_REAL = 2000
N_SPLITS_REAL = 500

SITES = {
    'US'       : ('PRO', 'BOS', 'CAM'),
    'SanBorja' : ('SBO', 'SNB', 'SBJ'),
    'Tsimane'  : ('NVM', 'MAJ', 'MAN', 'NUM', 'NUV', 'CVR'),
}
rng = np.random.default_rng(1)
groups = {}
for label, codes in SITES.items():
    groups[label] = load_group(
        BASE_DIR, codes, CONDITION,
        min_isi0_dprime=2.0, is_multi_isi=False,
        n_splits=N_SPLITS_REAL, rng=rng,
    )

In [ ]:
pairs = [('US','SanBorja'), ('US','Tsimane'), ('SanBorja','Tsimane')]
rows = []
for (a, b) in pairs:
    for dim, dim_label in [(1, 'participant'), (2, 'stimulus')]:
        res = bootstrap_intergroup_correlation_sem(
            groups[a], groups[b], trial_type=TRIAL,
            n_boot=N_BOOT_REAL, min_resp=MIN_RESP, bootstrap_dim=dim,
            reliability_mode='fixed', correct_atten=True,
            rng=np.random.default_rng(100 + dim), verbose=False,
        )
        for method in ['percentile', 'fisher_z', 'bca']:
            lo, hi = res.cis_corr[method]
            rows.append([f'{a}-{b}', dim_label, method, res.point_corr, lo, hi, hi-lo])

df = pd.DataFrame(rows, columns=['pair','boot_unit','method','r_corr','ci_lo','ci_hi','width'])
df.round(3)

In [ ]:
# Plot: 3 pairs x 2 bootstrap units, three CI methods overlaid
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
method_colors = {'percentile': '#888', 'fisher_z': '#4060b0', 'bca': '#b04040'}
method_offsets = {'percentile': -0.18, 'fisher_z': 0.0, 'bca': 0.18}

for ax, (a, b) in zip(axes, pairs):
    sub = df[df['pair'] == f'{a}-{b}']
    point = sub['r_corr'].iloc[0]
    for x_i, dim_label in enumerate(['participant', 'stimulus']):
        for method in ['percentile', 'fisher_z', 'bca']:
            row = sub[(sub['boot_unit'] == dim_label) & (sub['method'] == method)].iloc[0]
            x = x_i + method_offsets[method]
            ax.errorbar([x], [point],
                        yerr=[[point - row['ci_lo']], [row['ci_hi'] - point]],
                        fmt='o', color=method_colors[method], capsize=4,
                        label=method if x_i == 0 else None)
    ax.axhline(0, color='k', lw=0.6, alpha=0.5)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['participant boot', 'stimulus boot'])
    ax.set_title(f'r({a}, {b})  (atten-corrected)')
    if ax is axes[0]:
        ax.set_ylabel('intergroup r (corrected)')
        ax.legend(loc='lower right', fontsize=9)

plt.suptitle(f'{CONDITION} | {TRIAL}: CI methods side by side', y=1.02)
plt.tight_layout(); plt.show()

**Reading this plot.**

- Each panel is one pair. Each x-tick is a resampling unit (participant or stimulus). Each tick shows three error bars overlaid: percentile (gray), Fisher-z (blue), BCa (red).
- **If the three colors are similar at a given tick**: the bootstrap is well-behaved and the percentile CI is fine.
- **If BCa shifts noticeably from percentile**: the bootstrap is biased; trust BCa.
- **If participant and stimulus bars are very different lengths**: one source of variance dominates. Participant-dominated = your conclusions hinge on who you sampled; stimulus-dominated = on which sounds you chose. Both are real findings.

For the paper, recommended report: BCa CI for the participant bootstrap as the headline, with a note about the stimulus bootstrap as a robustness check.